# Simulation: Custom ETF Buy & Hold

This notebook simulates the **Custom ETF Buy & Hold Strategy**.

- **Portfolio:** 50% SPY, 30% QQQ, 20% VTI
- **Strategy:** Buy on day 1 and never sell. Add $500 every 10 trading days and buy more shares across all three ETFs using the custom distribution.
- **Initial Investment:** $10,000
- **Biweekly Contribution:** Add $500 every 10 trading days and buy more shares.

### How Buy & Hold Works
- Day 1: Invest $10,000 → buy SPY shares
- Day 10: Add $500 and buy more shares -> 50% SPY, 30% QQQ, 20% VTI
- Day 20: Add $500 and buy more shares -> 50% SPY, 30% QQQ, 20% VTI
- Day 30: Add $500 and buy buy more shares -> 50% SPY, 30% QQQ, 20% VTI
... 

---

## 1. Import Libraries, Configure Settings & Initialize Variables

In [11]:
import pandas as pd
import os

PROCESSED_DIR   = "../data/processed"
SIMULATIONS_DIR = "../data/simulations"
os.makedirs(SIMULATIONS_DIR, exist_ok=True)
INPUT_FILE  = os.path.join(PROCESSED_DIR, "prices_with_indicators.csv")
OUTPUT_FILE = os.path.join(SIMULATIONS_DIR, "custom_buy_hold.csv")

initial_investment = 10000 # Starting investment on day 1
contribution_amount = 500 # Amount added every 10 trading days
contribution_interval = 10 # Every 10 trading days (biweekly)

# weight =  percentage of money going into each ETF expressed as a decimal
weight_spy = 0.5
weight_qqq = 0.3
weight_vti = 0.2

# Strategy/portfolio labels 
STRATEGY = "Buy & Hold"
PORTFOLIO_TYPE = "Custom"

print(f"Input: {os.path.abspath(INPUT_FILE)}")
print(f"Output: {os.path.abspath(OUTPUT_FILE)}")
print(f"\nSimulation Parameters:")
print(f"Initial Investment: ${initial_investment:,}")
print(f"Contribution Amount: ${contribution_amount}")
print(f"Contribution Interval: Contributions are made every {contribution_interval} trading days")
print(f"Portfolio: SPY={weight_spy*100}%, QQQ={weight_qqq*100}%, VTI={weight_vti*100}%")
print(f"Strategy: {STRATEGY}")
print(f"Portfolio Type: {PORTFOLIO_TYPE}")

Input: c:\Users\Dell\Desktop\COMP3250-Project\data\processed\prices_with_indicators.csv
Output: c:\Users\Dell\Desktop\COMP3250-Project\data\simulations\custom_buy_hold.csv

Simulation Parameters:
Initial Investment: $10,000
Contribution Amount: $500
Contribution Interval: Contributions are made every 10 trading days
Portfolio: SPY=50.0%, QQQ=30.0%, VTI=20.0%
Strategy: Buy & Hold
Portfolio Type: Custom


## 2. Load `prices_with_indicators.csv`
Note:
- The first 49 rows have `NaN` in the MA50 columns — this is expected and correct.  
- Buy & Hold ignores MA50 entirely so these NaN values do not affect the simulation.

In [12]:
df = pd.read_csv(INPUT_FILE, parse_dates=["Date"])

# Confirm the file loaded correctly
print(f"Loaded prices_with_indicators.csv")
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Nulls: {df.isnull().sum().to_dict()}")
print(f"\nFirst 5 rows:")
df.head()

Loaded prices_with_indicators.csv
Rows: 6217
Columns: ['Date', 'SPY_Price', 'QQQ_Price', 'VTI_Price', 'SPY_MA50', 'QQQ_MA50', 'VTI_MA50']
Date range: 2001-06-15 to 2026-03-06
Nulls: {'Date': 0, 'SPY_Price': 0, 'QQQ_Price': 0, 'VTI_Price': 0, 'SPY_MA50': 49, 'QQQ_MA50': 49, 'VTI_MA50': 49}

First 5 rows:


,Date,SPY_Price,QQQ_Price,VTI_Price,SPY_MA50,QQQ_MA50,VTI_MA50
0,2001-06-15,77.995598,35.981022,36.027668,NaN,NaN,NaN
1,2001-06-18,77.617950,35.499580,35.797894,NaN,NaN,NaN
2,2001-06-19,77.957207,35.322208,35.898216,NaN,NaN,NaN
3,2001-06-20,78.366867,36.124603,36.276844,NaN,NaN,NaN
4,2001-06-21,79.256599,36.614502,36.568089,NaN,NaN,NaN


## 3. Simulate Custom ETF Buy & Hold Strategy

The simulation loops through each row through every trading day in the dataset.  

**Row 0: The Initial Investment:**
- cash = $10,000
- total_invested = $10,000
- cash = 0

**Every row where i > 0 and i % 10 == 0 (Contribution Days):**
- contribution = $500
- cash += $500
- total_invested += $500
- shares_spy += (cash * 0.5) / SPY_Price
- shares_qqq += (cash * 0.3) / QQQ_Price
- shares_vti += (cash * 0.2) / VTI_Price
- cash = 0

**All other rows:**
- contribution = 0
- cash = 0  
- shares_spy = same as previous row

Portfolio_Value = (shares_spy * SPY_Price) + cash

In [13]:
shares_spy = 0.0
shares_qqq = 0.0
shares_vti = 0.0
cash = 0.0
total_invested = 0.0

results = []

for i, row in df.iterrows():
    spy_price = row["SPY_Price"]
    qqq_price = row["QQQ_Price"]
    vti_price = row["VTI_Price"]

    contribution = 0.0  
    
    # Initial Investment 
    if i == 0:
        cash = initial_investment
        total_invested = initial_investment

    elif i % contribution_interval == 0:
        contribution = contribution_amount
        cash += contribution
        total_invested += contribution


    if cash > 0:
        
        original_cash = cash
        
        
        # SPY: 50% of cash
        amount_spy  = original_cash * weight_spy
        shares_spy += amount_spy / spy_price
        cash -= amount_spy
        
        # QQQ: 30% of cash
        amount_qqq  = original_cash * weight_qqq
        shares_qqq += amount_qqq / qqq_price
        cash -= amount_qqq  

        # VTI: 20% of available cash
        amount_vti  = original_cash * weight_vti
        shares_vti += amount_vti / vti_price
        cash -= amount_vti 
        

        # Cash goes to 0 after all shares are bought
        cash = round(cash, 10)
        
    portfolio_value = (
        (shares_spy * spy_price) +
        (shares_qqq * qqq_price) +
        (shares_vti * vti_price) +
        cash 
    )

    results.append({
        "Date": row["Date"],
        "SPY_Price": spy_price,
        "QQQ_Price": qqq_price,
        "VTI_Price": vti_price,
        "SPY_MA50": None,
        "QQQ_MA50": None,      
        "VTI_MA50": None,     
        "Position_SPY": 1, 
        "Position_QQQ": 1,
        "Position_VTI": 1, 
        "Contribution": contribution,
        "Cash": cash,  
        "Shares_SPY": shares_spy,
        "Shares_QQQ": shares_qqq, 
        "Shares_VTI": shares_vti,
        "Total_Invested": total_invested,
        "Portfolio_Value":portfolio_value,
        "Earnings": None,  
        "Daily_Return": None, 
        "Strategy": STRATEGY,
        "Portfolio_Type": PORTFOLIO_TYPE
    })

result_df = pd.DataFrame(results)
print(f"Simulation complete. Rows processed: {len(result_df)}")

Simulation complete. Rows processed: 6217


## 4. Calculate Earnings and Daily Return

- Earnings = Portfolio_Value - Total_Invested

- Daily_Return ( percentage change in portfolio value from the previous day)
- = Portfolio_Value.pct_change()
= (today - yesterday) / yesterday

- The first row of `Daily_Return` will always be `NaN` because there is no previous day to compare to.

In [14]:
result_df["Earnings"] = result_df["Portfolio_Value"] - result_df["Total_Invested"]
result_df["Daily_Return"] = result_df["Portfolio_Value"].pct_change()

print("Earnings and Daily Return calculated.")
print(result_df[["Date", "Portfolio_Value", "Total_Invested", "Earnings", "Daily_Return"]]
      .head(15).to_string(index=False))

Earnings and Daily Return calculated.
      Date  Portfolio_Value  Total_Invested    Earnings  Daily_Return
2001-06-15     10000.000000           10000    0.000000           NaN
2001-06-18      9922.893736           10000  -77.106264     -0.007711
2001-06-19      9935.422571           10000  -64.577429      0.001263
2001-06-20     10049.604561           10000   49.604561      0.011492
2001-06-21     10163.656153           10000  163.656153      0.011349
2001-06-22     10103.878131           10000  103.878131     -0.005882
2001-06-25     10052.285969           10000   52.285969     -0.005106
2001-06-26     10050.650362           10000   50.650362     -0.000163
2001-06-27     10067.677300           10000   67.677300      0.001694
2001-06-28     10159.098807           10000  159.098807      0.009081
2001-06-29     10784.160566           10500  284.160566      0.061527
2001-07-02     10837.302074           10500  337.302074      0.004928
2001-07-03     10848.880408           10500  348.880

## 5. Save `custom_buy_hold.csv`

Save the simulation results to `data/simulations/`.  
The file will have exactly 21 columns as defined in the data dictionary.

In [15]:
final_cols = [
    "Date", "SPY_Price", "QQQ_Price", "VTI_Price",
    "SPY_MA50", "QQQ_MA50", "VTI_MA50",
    "Position_SPY", "Position_QQQ", "Position_VTI",
    "Contribution", "Cash",
    "Shares_SPY", "Shares_QQQ", "Shares_VTI",
    "Total_Invested", "Portfolio_Value", "Earnings",
    "Daily_Return", "Strategy", "Portfolio_Type"
]

result_df = result_df[final_cols]
result_df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved custom_buy_hold.csv")
print(f"Path: {os.path.abspath(OUTPUT_FILE)}")
print(f"Rows: {len(result_df)}")
print(f"Columns: {len(result_df.columns)} —> {list(result_df.columns)}")
print(f"\nFinal Portfolio Summary")
print(f"Total Invested: ${result_df['Total_Invested'].iloc[-1]:,.2f}")
print(f"Final Value: ${result_df['Portfolio_Value'].iloc[-1]:,.2f}")
print(f"Total Earnings: ${result_df['Earnings'].iloc[-1]:,.2f}")

Saved custom_buy_hold.csv
Path: c:\Users\Dell\Desktop\COMP3250-Project\data\simulations\custom_buy_hold.csv
Rows: 6217
Columns: 21 —> ['Date', 'SPY_Price', 'QQQ_Price', 'VTI_Price', 'SPY_MA50', 'QQQ_MA50', 'VTI_MA50', 'Position_SPY', 'Position_QQQ', 'Position_VTI', 'Contribution', 'Cash', 'Shares_SPY', 'Shares_QQQ', 'Shares_VTI', 'Total_Invested', 'Portfolio_Value', 'Earnings', 'Daily_Return', 'Strategy', 'Portfolio_Type']

Final Portfolio Summary
Total Invested: $320,500.00
Final Value: $2,210,672.43
Total Earnings: $1,890,172.43
